# Madonna Sonic Atlas — v2
### Data-driven audio DNA analysis of Madonna's own catalog

**Rebuild goals (v1 → v2):**
- Fix broken references (`madonna_id`), remove duplicated blocks, dedupe redundant exports
- Use the full audio-feature set instead of 5 cherry-picked features
- Fix K at 4 (project requirement) and compare KMeans vs. GMM to pick whichever actually fits the data better, instead of assuming KMeans
- Report *cluster stability*, not just one lucky `random_state`
- Auto-generate cluster personas from the data (z-scores) instead of hand-typed guesses
- One clean, deduplicated export pass at the end (no more `kmeans_cluster_0_all_tracks.csv` x4 style sprawl)
- Dropped the multi-artist diva comparison — this version is scoped to Madonna's catalog only


## 1. Setup

In [ ]:
# UMAP is not preinstalled on Kaggle by default in every image — install quietly if missing
try:
    import umap
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"])
    import umap


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import RobustScaler, PowerTransformer
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# --- Brand palette (kept from v1 — Madonna pink/gold works, just applied more consistently) ---
PALETTE = ['#FF1493', '#1DB954', '#FFD700', '#7B2CBF', '#00B4D8']
sns.set_theme(style="whitegrid", rc={
    "axes.edgecolor": "#333333",
    "axes.titleweight": "bold",
    "figure.facecolor": "white",
})
plt.rcParams['font.size'] = 10

RANDOM_STATE = 42


## 2. Load data

Same source as v1 (`yamaerenay/spotify-dataset-19212020-600k-tracks`), but with an explicit
existence check so a missing/renamed Kaggle dataset path fails loudly instead of silently
producing an empty frame three cells later.

In [ ]:
DATA_DIR = "/kaggle/input/datasets/yamaerenay/spotify-dataset-19212020-600k-tracks"

artists = pd.read_csv(f"{DATA_DIR}/artists.csv")
tracks = pd.read_csv(f"{DATA_DIR}/tracks.csv")

print(f"artists: {artists.shape}, tracks: {tracks.shape}")
assert 'id_artists' in tracks.columns and 'artists' in tracks.columns, "Unexpected schema — check dataset version"


## 3. Extract & clean Madonna's catalog

v1 bug: `madonna_id` was printed but only `madonna_id_str` was ever defined — leftover from an
earlier variable rename. Fixed here, plus de-duplication (the raw dataset has near-duplicate
entries for reissues/remasters that inflate cluster counts without adding signal).

In [ ]:
MADONNA_ID = '6tbjWDEIzxoDsBA1FuhfPW'

madonna_tracks = tracks[tracks['id_artists'].str.contains(MADONNA_ID, na=False)].copy()
print(f"Raw Madonna tracks: {madonna_tracks.shape}")

madonna_cleaned = madonna_tracks.copy()
madonna_cleaned['release_year'] = pd.to_datetime(madonna_cleaned['release_date'], errors='coerce').dt.year

# de-dupe: same title + duration within 2s is almost certainly the same recording (remaster/reissue)
madonna_cleaned['duration_bucket'] = (madonna_cleaned['duration_ms'] / 2000).round()
madonna_cleaned = (madonna_cleaned
                   .sort_values('popularity', ascending=False)
                   .drop_duplicates(subset=['name', 'duration_bucket'])
                   .drop(columns='duration_bucket'))

madonna_cleaned = madonna_cleaned.dropna(subset=['release_year'])

print(f"After de-dup + year cleanup: {madonna_cleaned.shape}")
print(f"Year range: {madonna_cleaned['release_year'].min():.0f} - {madonna_cleaned['release_year'].max():.0f}")
print(f"Missing values:\n{madonna_cleaned.isnull().sum()[madonna_cleaned.isnull().sum() > 0]}")


## 4. EDA: feature distributions & correlation

Before picking a feature set for clustering, check what's actually redundant. v1 used
`['danceability', 'energy', 'valence', 'acousticness', 'loudness']` with no justification —
here we look at the full audio-feature correlation matrix first.

In [ ]:
AUDIO_FEATURES_FULL = ['danceability', 'energy', 'loudness', 'speechiness',
                        'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

corr = madonna_cleaned[AUDIO_FEATURES_FULL].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[0],
            linewidths=0.5, cbar_kws={'label': 'Pearson r'})
axes[0].set_title('Audio Feature Correlation')

madonna_cleaned[AUDIO_FEATURES_FULL].boxplot(ax=axes[1], rot=45)
axes[1].set_title('Feature Distributions (raw scale)')

plt.tight_layout()
plt.savefig('eda_features.png', dpi=100, bbox_inches='tight')
plt.show()

print("Note: loudness and energy are typically correlated (~0.6-0.8) but not redundant enough")
print("to drop either — loudness captures mastering/production era, energy captures perceived intensity.")


## 5. Feature set & scaling

Expanding from 5 → 9 audio features. `instrumentalness`, `speechiness`, and `liveness` are
heavily right-skewed — most tracks sit near 0, with a handful of true outliers (spoken intros,
live recordings, instrumental interludes) far out on the tail. A scaler that only fixes *scale*
(like `RobustScaler`) doesn't fix that *shape* — those few outlier tracks stay just as far from
the main mass in scaled space, and a clustering algorithm happily peels them off into their own
tiny cluster instead of finding meaningful structure across the catalog. Using `PowerTransformer`
(Yeo-Johnson) instead reshapes each feature toward a roughly Gaussian distribution first, which
pulls those outliers back in relative to the bulk of the data and gives clustering something more
balanced to work with.


In [ ]:
CLUSTER_FEATURES = AUDIO_FEATURES_FULL  # all 9, justified by the correlation check above

X_cluster = madonna_cleaned[CLUSTER_FEATURES].copy()
scaler = PowerTransformer(method='yeo-johnson')
X_scaled = scaler.fit_transform(X_cluster)

print(f"Features used: {CLUSTER_FEATURES}")
print(f"Scaled shape: {X_scaled.shape}")


## 6. Choosing K — diagnostics, but K is fixed at 4

Elbow (inertia), silhouette, and GMM's BIC/AIC are still computed across a range of K, purely as
diagnostic context — they tell you how well-supported a given K is by the data. The final K used
everywhere below is fixed at **4** (project requirement), not picked automatically from these
curves. The prints at the end of the next cell show how K=4 compares to whatever the metrics
would have picked on their own, so you can see whether 4 is a reasonable choice or a compromise.


In [ ]:
K_range = range(2, 11)
inertias, silhouettes, db_scores, bics, aics = [], [], [], [], []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))
    db_scores.append(davies_bouldin_score(X_scaled, km.labels_))

    gmm = GaussianMixture(n_components=k, covariance_type='diag', random_state=RANDOM_STATE, n_init=5).fit(X_scaled)
    bics.append(gmm.bic(X_scaled))
    aics.append(gmm.aic(X_scaled))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(K_range, inertias, 'o-', color=PALETTE[0])
axes[0].axvline(4, ls='--', color='gray', alpha=0.6)
axes[0].set(xlabel='K', ylabel='Inertia', title='Elbow (KMeans)')

axes[1].plot(K_range, silhouettes, 's-', color=PALETTE[1])
best_sil_k = list(K_range)[int(np.argmax(silhouettes))]
axes[1].axvline(4, ls='--', color='gray', alpha=0.6)
axes[1].set(xlabel='K', ylabel='Silhouette', title=f'Silhouette (best K={best_sil_k}, using K=4)')

axes[2].plot(K_range, bics, '^-', label='BIC', color=PALETTE[2])
axes[2].plot(K_range, aics, 'v-', label='AIC', color=PALETTE[3])
best_bic_k = list(K_range)[int(np.argmin(bics))]
axes[2].axvline(4, ls='--', color='gray', alpha=0.6)
axes[2].set(xlabel='K', ylabel='Score', title=f'GMM BIC/AIC (best K={best_bic_k}, using K=4)')
axes[2].legend()

plt.tight_layout()
plt.savefig('k_selection.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Silhouette suggests K={best_sil_k} (score={max(silhouettes):.3f})")
print(f"GMM BIC suggests K={best_bic_k}")
print("Be honest about scale: silhouette scores around 0.2-0.3 are normal for a single artist's")
print("catalog — audio features overlap a lot within one discography. This is not a failure,")
print("it just means clusters are 'soft neighborhoods' rather than hard-separated genres.")

OPTIMAL_K = 4  # fixed per project requirement, not the algorithmic argmax above
k4_idx = list(K_range).index(4)
print(f"\nFinal K fixed at {OPTIMAL_K} (silhouette at K=4: {silhouettes[k4_idx]:.3f}, "
      f"vs. best silhouette K={best_sil_k} at {max(silhouettes):.3f})")


### 6b. Is K=4 stable, or just acceptable?

Section 6 shows K=4 isn't the silhouette/BIC argmax — it's fixed by project requirement. That's a
different question from whether K=4 produces a *stable* structure. A K that scores acceptably but
reshuffles heavily across seeds is weaker evidence than one that scores similarly but stays put.
This repeats the seed-stability check from 7b.2 across each candidate K, before the final model is
even fit, so K=4's stability can be judged against its neighbors.

In [ ]:
K_stability_range = range(2, 8)  # narrower than the K=2-11 diagnostic range — enough to see the trend around K=4
n_stability_seeds = 15
k_stability_results = {}

for k in K_stability_range:
    labels_by_seed = [KMeans(n_clusters=k, n_init=10, random_state=s).fit_predict(X_scaled)
                       for s in range(1, n_stability_seeds + 1)]
    aris = [adjusted_rand_score(labels_by_seed[i], labels_by_seed[j])
            for i in range(n_stability_seeds) for j in range(i + 1, n_stability_seeds)]
    k_stability_results[k] = (np.mean(aris), np.std(aris))
    marker = '  <-- used' if k == 4 else ''
    print(f"K={k}: mean ARI={np.mean(aris):.3f}, std={np.std(aris):.3f}{marker}")

fig, ax = plt.subplots(figsize=(7, 4))
ks = list(k_stability_results.keys())
means = [k_stability_results[k][0] for k in ks]
stds = [k_stability_results[k][1] for k in ks]
ax.errorbar(ks, means, yerr=stds, fmt='o-', color=PALETTE[0], capsize=4)
ax.axvline(4, ls='--', color='gray', alpha=0.6, label='K=4 (used)')
ax.set(xlabel='K', ylabel='Mean seed-to-seed ARI', title='Clustering stability by K')
ax.legend()
plt.tight_layout()
plt.savefig('k_stability.png', dpi=100, bbox_inches='tight')
plt.show()

k4_rank = sorted(k_stability_results, key=lambda k: -k_stability_results[k][0]).index(4) + 1
print(f"\nK=4 ranks #{k4_rank} of {len(K_stability_range)} candidate K values by stability.")


## 7. Fit KMeans + GMM at K=4, keep whichever fits better

Both KMeans and a GMM are fit at K=4. Rather than assuming KMeans is "the" clustering algorithm,
the two are compared on silhouette score and Davies-Bouldin index, and the labels actually used
downstream (`madonna_cleaned['cluster']`) come from whichever model scores better — not from a
fixed pick. Adjusted Rand Index between the two label sets shows how much this choice even
matters: a high ARI means KMeans and GMM already agree, so picking the "better" one is close to
moot; a low ARI means the two disagree and the model choice actually changes the story. We also
bootstrap-resample the selected model to check run-to-run stability.


In [ ]:
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10).fit(X_scaled)
gmm = GaussianMixture(n_components=OPTIMAL_K, covariance_type='diag', random_state=RANDOM_STATE, n_init=5).fit(X_scaled)
gmm_labels = gmm.predict(X_scaled)

kmeans_gmm_ari = adjusted_rand_score(kmeans.labels_, gmm_labels)
print(f"KMeans vs GMM agreement (ARI): {kmeans_gmm_ari:.3f}  (1.0 = identical, 0.0 = random)")

candidates = {
    'KMeans': {
        'labels': kmeans.labels_,
        'silhouette': silhouette_score(X_scaled, kmeans.labels_),
        'davies_bouldin': davies_bouldin_score(X_scaled, kmeans.labels_),
    },
    'GMM': {
        'labels': gmm_labels,
        'silhouette': silhouette_score(X_scaled, gmm_labels),
        'davies_bouldin': davies_bouldin_score(X_scaled, gmm_labels),
    },
}

for name, metrics in candidates.items():
    sizes = pd.Series(metrics['labels']).value_counts()
    metrics['min_cluster_pct'] = float(sizes.min() / sizes.sum() * 100)
    print(f"{name:8s} silhouette={metrics['silhouette']:.3f}  davies_bouldin={metrics['davies_bouldin']:.3f}  "
          f"sizes={sorted(sizes.tolist(), reverse=True)} (smallest={metrics['min_cluster_pct']:.1f}%)")

CLUSTER_METHOD = max(candidates, key=lambda name: candidates[name]['silhouette'])
madonna_cleaned['cluster'] = candidates[CLUSTER_METHOD]['labels']
print(f"\nSelected model: {CLUSTER_METHOD} (higher silhouette wins)")
if candidates[CLUSTER_METHOD]['min_cluster_pct'] < 5:
    print("Warning: smallest cluster is under 5% of the catalog — likely a few outlier tracks")
    print("isolated into their own cluster rather than a real audio-DNA segment.")

# bootstrap stability: refit the *selected* method on 80% resamples, compare labels on the shared subset
n_boot = 15
boot_aris = []
rng = np.random.RandomState(RANDOM_STATE)
full_idx = np.arange(len(X_scaled))
final_labels = madonna_cleaned['cluster'].values

for _ in range(n_boot):
    sample_idx = rng.choice(full_idx, size=int(0.8 * len(full_idx)), replace=False)
    if CLUSTER_METHOD == 'KMeans':
        boot_model = KMeans(n_clusters=OPTIMAL_K, n_init=5, random_state=rng.randint(10000)).fit(X_scaled[sample_idx])
        boot_labels = boot_model.labels_
    else:
        boot_model = GaussianMixture(n_components=OPTIMAL_K, covariance_type='diag', n_init=3, random_state=rng.randint(10000)).fit(X_scaled[sample_idx])
        boot_labels = boot_model.predict(X_scaled[sample_idx])
    boot_aris.append(adjusted_rand_score(final_labels[sample_idx], boot_labels))

print(f"Bootstrap stability (mean ARI over {n_boot} resamples): {np.mean(boot_aris):.3f} ± {np.std(boot_aris):.3f}")
print(f"\nFinal silhouette: {candidates[CLUSTER_METHOD]['silhouette']:.3f}")
print(f"Final Davies-Bouldin: {candidates[CLUSTER_METHOD]['davies_bouldin']:.3f}")
print(f"\nCluster sizes:\n{madonna_cleaned['cluster'].value_counts().sort_index()}")


## 8. Auto-generated cluster personas

v1 hardcoded names like `"Energetic and Danceable"` and `"The Disco Dynamo"` that had to be
manually re-checked every time the clustering changed. Here the name is derived directly from
each cluster's z-scores vs. the full catalog — if you re-run this with a different K or feature
set, the labels update themselves instead of silently going stale.

In [ ]:
FEATURE_ADJECTIVES = {
    'danceability':     ('Danceable', 'Understated'),
    'energy':           ('Energetic', 'Mellow'),
    'valence':          ('Uplifting', 'Melancholic'),
    'acousticness':     ('Acoustic', 'Produced'),
    'loudness':         ('Bold', 'Subtle'),
    'speechiness':      ('Talkative', 'Melodic'),
    'instrumentalness': ('Instrumental', 'Vocal-driven'),
    'liveness':         ('Live-feel', 'Studio-polished'),
    'tempo':            ('Fast-paced', 'Slow-burn'),
}

global_mean = X_cluster.mean()
global_std = X_cluster.std()
cluster_means = madonna_cleaned.groupby('cluster')[CLUSTER_FEATURES].mean()
z_scores = (cluster_means - global_mean) / global_std

cluster_names = {}
for cid in range(OPTIMAL_K):
    top2 = z_scores.loc[cid].abs().sort_values(ascending=False).head(2).index
    labels = []
    for feat in top2:
        high, low = FEATURE_ADJECTIVES[feat]
        labels.append(high if z_scores.loc[cid, feat] > 0 else low)
    cluster_names[cid] = " & ".join(labels)

for cid, name in cluster_names.items():
    size = (madonna_cleaned['cluster'] == cid).sum()
    pct = size / len(madonna_cleaned) * 100
    print(f"Cluster {cid} — \"{name}\"  ({size} tracks, {pct:.1f}%)")


## 7b. Structural validation: is this clustering real or arbitrary?

The bootstrap ARI above answers "does resampling the *data* change the labels?" It doesn't
answer three other questions that matter more for an unsupervised, no-ground-truth project:

1. **Feature-set sensitivity** — is the cluster structure a property of Madonna's music, or an
   artifact of picking these particular 9 features?
2. **Seed-to-seed stability** — independent of resampling, how much does clustering vary purely
   from initialization randomness?
3. **Per-track silhouette** — does the *average* silhouette hide a cluster that's mostly negative,
   i.e. arguably not a real cluster at all, but an artifact of forcing K=4?

None of this is hyperparameter tuning — no step below is chosen to make a metric look better.
Each one is a diagnostic asking "would I have gotten a similar answer under a different reasonable
choice?" That's the right question for exploratory clustering with no labels to validate against.


### 7b.1 Feature-set sensitivity

Refit clustering under several different feature subsets and compare the resulting labels to the
main (9-feature) result via Adjusted Rand Index. High ARI across subsets means the structure isn't
an artifact of any one feature choice.

In [ ]:
from itertools import combinations

# Candidate feature sets to test against the full 9-feature result
feature_sets = {
    'all_9': AUDIO_FEATURES_FULL,
    'drop_correlated': [f for f in AUDIO_FEATURES_FULL if f not in ('energy', 'loudness')],  # energy/loudness are the most correlated pair from Sec. 4
    'core_5_v1': ['danceability', 'energy', 'valence', 'acousticness', 'loudness'],  # the original v1 feature set, for continuity
}

# Two random 6-feature subsets, seeded for reproducibility
rng_fs = np.random.RandomState(RANDOM_STATE)
all_combos = list(combinations(AUDIO_FEATURES_FULL, 6))
for i, idx in enumerate(rng_fs.choice(len(all_combos), size=2, replace=False)):
    feature_sets[f'random_6_v{i+1}'] = list(all_combos[idx])

# PCA-components variant: cluster on enough PCs to explain 90% variance instead of raw features
pca_fs = PCA(n_components=0.90, random_state=RANDOM_STATE).fit_transform(X_scaled)

feature_set_results = {}
for name, feats in feature_sets.items():
    X_fs = scaler.fit_transform(madonna_cleaned[feats])  # refit scaler per subset — same PowerTransformer, different columns
    labels_fs = KMeans(n_clusters=OPTIMAL_K, n_init=10, random_state=RANDOM_STATE).fit_predict(X_fs)
    ari_vs_main = adjusted_rand_score(final_labels, labels_fs)
    feature_set_results[name] = ari_vs_main
    print(f"{name:16s} (n_features={len(feats):2d})  ARI vs. main clustering: {ari_vs_main:.3f}")

# PCA variant fit separately since it isn't a named column subset
labels_pca_fs = KMeans(n_clusters=OPTIMAL_K, n_init=10, random_state=RANDOM_STATE).fit_predict(pca_fs)
ari_pca = adjusted_rand_score(final_labels, labels_pca_fs)
feature_set_results['pca_90pct_variance'] = ari_pca
print(f"{'pca_90pct_variance':16s} (n_components={pca_fs.shape[1]:2d})  ARI vs. main clustering: {ari_pca:.3f}")

mean_fs_ari = np.mean(list(feature_set_results.values()))
print(f"\nMean ARI across feature-set variants: {mean_fs_ari:.3f}")
if mean_fs_ari > 0.5:
    print("Structure is reasonably robust to feature-set choice — not an artifact of picking these 9 features.")
else:
    print("Structure is sensitive to feature-set choice — treat cluster identity as feature-set-dependent, not universal.")


### 7b.2 Seed-to-seed stability (independent of resampling)

The bootstrap analysis in Section 7 varies *which tracks* are in the training set. This is a
narrower test: keep the full dataset fixed and vary only the random seed controlling
initialization, isolating how much the algorithm's own randomness — separate from data
resampling — moves the labels.

In [ ]:
n_seeds = 50
seed_labels = []

for seed in range(1, n_seeds + 1):
    if CLUSTER_METHOD == 'KMeans':
        model = KMeans(n_clusters=OPTIMAL_K, n_init=10, random_state=seed).fit(X_scaled)
        seed_labels.append(model.labels_)
    else:
        model = GaussianMixture(n_components=OPTIMAL_K, covariance_type='diag', n_init=5, random_state=seed).fit(X_scaled)
        seed_labels.append(model.predict(X_scaled))

# Pairwise ARI across all seed pairs
pairwise_aris = []
for i in range(n_seeds):
    for j in range(i + 1, n_seeds):
        pairwise_aris.append(adjusted_rand_score(seed_labels[i], seed_labels[j]))

pairwise_aris = np.array(pairwise_aris)
print(f"Seed-to-seed stability ({CLUSTER_METHOD}, {n_seeds} seeds, {len(pairwise_aris)} pairs):")
print(f"  Mean ARI: {pairwise_aris.mean():.3f}")
print(f"  Std ARI:  {pairwise_aris.std():.3f}")
print(f"  Min ARI:  {pairwise_aris.min():.3f}")
print(f"  Max ARI:  {pairwise_aris.max():.3f}")


### 7b.3 Cluster stability matrix

A single mean ARI number can hide structure — e.g. most seed pairs might agree closely while a
handful of outlier seeds land on a very different partition. Visualizing the full pairwise matrix
for a sample of seeds makes that visible directly.

In [ ]:
n_show = 10  # first 10 of the 50 seeds, for a readable heatmap
stability_matrix = np.zeros((n_show, n_show))
for i in range(n_show):
    for j in range(n_show):
        stability_matrix[i, j] = adjusted_rand_score(seed_labels[i], seed_labels[j])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(stability_matrix, annot=True, fmt='.2f', cmap='viridis', vmin=0, vmax=1,
            xticklabels=[f'Seed {i+1}' for i in range(n_show)],
            yticklabels=[f'Seed {i+1}' for i in range(n_show)],
            ax=ax, cbar_kws={'label': 'ARI'})
ax.set_title(f'{CLUSTER_METHOD} pairwise stability (ARI), first {n_show} of {n_seeds} seeds')
plt.tight_layout()
plt.savefig('seed_stability_matrix.png', dpi=100, bbox_inches='tight')
plt.show()


### 7b.4 Per-track (per-cluster) silhouette distribution

The overall silhouette score reported in Section 7 is a single average. It can mask a cluster that
is mostly well-separated alongside one that is barely distinguishable from its neighbors — which
matters here specifically because K=4 was fixed as a project requirement, not chosen by the data
(Section 6).

In [ ]:
from sklearn.metrics import silhouette_samples

sample_silhouettes = silhouette_samples(X_scaled, final_labels)

fig, ax = plt.subplots(figsize=(9, 5))
cluster_sil_means = {}
for cid in range(OPTIMAL_K):
    vals = sample_silhouettes[final_labels == cid]
    cluster_sil_means[cid] = vals.mean()
    ax.hist(vals, bins=20, alpha=0.6, label=f'Cluster {cid} — "{cluster_names[cid]}" (mean={vals.mean():.2f})',
            color=PALETTE[cid % len(PALETTE)])

ax.axvline(0, ls='--', color='black', alpha=0.5, label='Silhouette = 0 (no separation)')
ax.set(xlabel='Silhouette coefficient', ylabel='Track count',
       title='Per-track silhouette distribution by cluster')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('per_track_silhouette.png', dpi=100, bbox_inches='tight')
plt.show()

print("Per-cluster mean silhouette:")
for cid, mean_sil in sorted(cluster_sil_means.items(), key=lambda x: -x[1]):
    frac_negative = (sample_silhouettes[final_labels == cid] < 0).mean() * 100
    print(f"  Cluster {cid} (\"{cluster_names[cid]}\"): mean={mean_sil:.3f}, "
          f"{frac_negative:.1f}% of tracks have negative silhouette")

weak_clusters = [cid for cid, m in cluster_sil_means.items() if m < 0.1]
if weak_clusters:
    print(f"\nCluster(s) {weak_clusters} have weak mean silhouette (<0.10) — likely reflect")
    print("K=4 being a fixed project requirement rather than a data-driven optimum, per Section 6.")


## 9. Dimensionality reduction: PCA + UMAP

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_scaled)
cumsum_var = np.cumsum(pca.explained_variance_ratio_)

reducer = umap.UMAP(n_neighbors=15, min_dist=0.3, random_state=RANDOM_STATE)
X_umap = reducer.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_, alpha=0.6)
axes[0].plot(range(1, len(cumsum_var) + 1), cumsum_var, 'o-', color=PALETTE[0])
axes[0].axhline(0.95, ls='--', color='gray')
axes[0].set(xlabel='Component', ylabel='Variance Ratio', title='PCA Explained Variance')

for ax, (X_dr, name) in zip(axes[1:], [(X_pca, 'PCA'), (X_umap, 'UMAP')]):
    for cid in range(OPTIMAL_K):
        mask = madonna_cleaned['cluster'] == cid
        ax.scatter(X_dr[mask.values, 0], X_dr[mask.values, 1], s=90, alpha=0.75,
                   edgecolors='white', linewidth=0.5, color=PALETTE[cid % len(PALETTE)],
                   label=cluster_names[cid])
    ax.set(xlabel=f'{name} 1', ylabel=f'{name} 2', title=f'{name} projection')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('dimensionality_reduction.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"PCA: first 2 PCs explain {cumsum_var[1]*100:.1f}% of variance")
print("UMAP preserves both local neighborhoods and global cluster shape better than t-SNE")
print("for a dataset this small (~hundreds of points) — using it in place of v1's t-SNE.")


## 9b. Sensitivity checks: GMM covariance assumption & UMAP topology

Two remaining checks, both framed as sensitivity analysis rather than tuning — neither is chosen
to make a metric look better:

- **GMM covariance type**: Section 7 fixes `covariance_type='diag'`, a real modeling assumption
  (each feature has its own variance but features are uncorrelated within a cluster) that directly
  shapes cluster geometry. Worth checking whether looser or tighter assumptions change the picture.
- **UMAP `n_neighbors`**: UMAP here is a visualization layer, not the clustering objective itself.
  It's deliberately *not* tuned against silhouette — doing so would mean picking the projection
  that makes clusters look best and then citing that same picture as evidence clusters are good,
  which is circular. Instead, check whether the layout's core topology holds across neighborhood
  sizes.

### 9b.1 GMM covariance-type comparison

In [ ]:
covariance_types = ['diag', 'full', 'tied', 'spherical']
gmm_cov_results = {}

for cov_type in covariance_types:
    gmm_cov = GaussianMixture(n_components=OPTIMAL_K, covariance_type=cov_type,
                               n_init=5, random_state=RANDOM_STATE).fit(X_scaled)
    labels_cov = gmm_cov.predict(X_scaled)
    sizes = pd.Series(labels_cov).value_counts()
    gmm_cov_results[cov_type] = {
        'bic': gmm_cov.bic(X_scaled),
        'aic': gmm_cov.aic(X_scaled),
        'silhouette': silhouette_score(X_scaled, labels_cov),
        'min_cluster_pct': float(sizes.min() / sizes.sum() * 100),
        'ari_vs_diag': adjusted_rand_score(gmm_labels, labels_cov),  # gmm_labels = the diag fit from Section 7
    }

print(f"{'covariance':10s} {'BIC':>10s} {'AIC':>10s} {'silhouette':>11s} {'min_cluster%':>13s} {'ARI vs diag':>12s}")
for cov_type, m in gmm_cov_results.items():
    print(f"{cov_type:10s} {m['bic']:10.1f} {m['aic']:10.1f} {m['silhouette']:11.3f} "
          f"{m['min_cluster_pct']:12.1f}% {m['ari_vs_diag']:12.3f}")

best_bic_cov = min(gmm_cov_results, key=lambda c: gmm_cov_results[c]['bic'])
print(f"\nLowest BIC: '{best_bic_cov}'. Current pipeline uses 'diag' — kept for parameter economy")
print("(full covariance has O(features^2) parameters per cluster, which risks overfitting on a")
print("catalog this size); ARI-vs-diag above shows how much that choice actually moves the labels.")


### 9b.2 UMAP topology sensitivity

In [ ]:
umap_neighbor_values = [5, 15, 30, 50]
umap_projections = {}

for n_neighbors in umap_neighbor_values:
    reducer_test = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.3, random_state=RANDOM_STATE)
    umap_projections[n_neighbors] = reducer_test.fit_transform(X_scaled)

fig, axes = plt.subplots(1, len(umap_neighbor_values), figsize=(20, 4.5))
for ax, n_neighbors in zip(axes, umap_neighbor_values):
    X_dr = umap_projections[n_neighbors]
    for cid in range(OPTIMAL_K):
        mask = (madonna_cleaned['cluster'] == cid).values
        ax.scatter(X_dr[mask, 0], X_dr[mask, 1], s=50, alpha=0.75,
                   edgecolors='white', linewidth=0.4, color=PALETTE[cid % len(PALETTE)])
    ax.set(xlabel='UMAP 1', ylabel='UMAP 2', title=f'n_neighbors={n_neighbors}')

plt.tight_layout()
plt.savefig('umap_neighbor_sensitivity.png', dpi=100, bbox_inches='tight')
plt.show()

# Quantify topology agreement: does a KNN classifier trained on one projection's neighborhoods
# agree with another's, for the same cluster labels? Simple proxy: correlation of pairwise distances.
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr

base_dists = pdist(umap_projections[15])  # the n_neighbors=15 value used in Section 9
print("Pairwise-distance rank correlation vs. the n_neighbors=15 layout used in Section 9:")
for n_neighbors in umap_neighbor_values:
    if n_neighbors == 15:
        continue
    corr, _ = spearmanr(base_dists, pdist(umap_projections[n_neighbors]))
    print(f"  n_neighbors={n_neighbors}: Spearman r={corr:.3f}")

print("\nNote: n_neighbors was not selected by which layout looks best — this is a robustness")
print("check on the n_neighbors=15 choice already used for visualization in Section 9, not a search")
print("for a better one.")


## 10. Cluster dashboard (heatmap + radar + timeline)

One combined figure instead of v1's three separate near-duplicate plotting blocks scattered
across cells 8, 11, and 13.

In [ ]:
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, OPTIMAL_K, hspace=0.4, wspace=0.35)

ax_heat = fig.add_subplot(gs[0, :OPTIMAL_K // 2 + 1])
sns.heatmap(cluster_means.T, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax_heat,
            linewidths=0.5, linecolor='white', cbar_kws={'label': 'Mean value'})
ax_heat.set_xticklabels([cluster_names[i] for i in range(OPTIMAL_K)], rotation=20, ha='right')
ax_heat.set_title('Cluster Audio Profiles', fontweight='bold')

ax_pie = fig.add_subplot(gs[0, OPTIMAL_K // 2 + 1:])
sizes = madonna_cleaned['cluster'].value_counts().sort_index()
ax_pie.pie(sizes, labels=[cluster_names[i] for i in sizes.index], autopct='%1.0f%%',
           colors=[PALETTE[i % len(PALETTE)] for i in sizes.index], startangle=90,
           textprops={'fontsize': 8})
ax_pie.set_title('Cluster Distribution', fontweight='bold')

radar_feats = ['danceability', 'energy', 'valence', 'acousticness', 'loudness']
radar_norm = (cluster_means[radar_feats] - X_cluster[radar_feats].min()) / (
    X_cluster[radar_feats].max() - X_cluster[radar_feats].min())
angles = np.linspace(0, 2 * np.pi, len(radar_feats), endpoint=False).tolist()
angles += angles[:1]

for cid in range(OPTIMAL_K):
    ax = fig.add_subplot(gs[1, cid], projection='polar')
    vals = radar_norm.loc[cid].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, color=PALETTE[cid % len(PALETTE)])
    ax.fill(angles, vals, alpha=0.25, color=PALETTE[cid % len(PALETTE)])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_feats, size=7)
    ax.set_ylim(0, 1)
    ax.set_title(cluster_names[cid], fontsize=9, pad=15)

plt.savefig('cluster_dashboard.png', dpi=100, bbox_inches='tight')
plt.show()


## 11. Track recommendations (content-based similarity)

Kept from v1, unchanged logic — cosine similarity on the full feature set was already sound.

In [ ]:
SIM_FEATURES = ['danceability', 'energy', 'valence', 'acousticness', 'loudness',
                 'speechiness', 'instrumentalness', 'liveness']
X_sim = RobustScaler().fit_transform(madonna_cleaned[SIM_FEATURES])
sim_matrix = cosine_similarity(X_sim)

recommendations = {}
for idx in range(len(madonna_cleaned)):
    similar_idx = np.argsort(sim_matrix[idx])[-6:-1][::-1]
    track_name = madonna_cleaned.iloc[idx]['name']
    recommendations[track_name] = [madonna_cleaned.iloc[i]['name'] for i in similar_idx]

print(f"Generated recommendations for {len(recommendations)} tracks")


## 12. Consolidated export

v1 wrote ~15 separate CSV/JSON files with overlapping content (e.g. `cluster_profiles.json`,
`kmeans_cluster_profiles.csv`, and `madonna_metadata.json` all contained the same numbers in
different shapes). One export pass, one folder, clear naming.

In [ ]:
import os
os.makedirs('exports', exist_ok=True)

# 1. Full track-level dataset
export_cols = ['name', 'artists', 'release_year', 'popularity', 'cluster'] + CLUSTER_FEATURES
export_df = madonna_cleaned[export_cols].copy()
export_df['cluster_name'] = export_df['cluster'].map(cluster_names)
export_df.to_csv('exports/madonna_tracks_clustered.csv', index=False)

# 2. Cluster summary
cluster_summary = cluster_means.copy()
cluster_summary['cluster_name'] = cluster_summary.index.map(cluster_names)
cluster_summary['size'] = madonna_cleaned['cluster'].value_counts().sort_index()
cluster_summary.to_csv('exports/cluster_summary.csv')

# 3. Recommendations
with open('exports/track_recommendations.json', 'w') as f:
    json.dump(recommendations, f, indent=2)

# 4. Everything a web frontend needs, in one JSON
web_bundle = {
    'metadata': {
        'total_tracks': int(len(madonna_cleaned)),
        'year_range': [int(madonna_cleaned['release_year'].min()), int(madonna_cleaned['release_year'].max())],
        'clustering': {
            'method': CLUSTER_METHOD, 'k': int(OPTIMAL_K),
            'silhouette': float(candidates[CLUSTER_METHOD]['silhouette']),
            'davies_bouldin': float(candidates[CLUSTER_METHOD]['davies_bouldin']),
            'kmeans_gmm_ari': float(kmeans_gmm_ari),
            'bootstrap_stability_ari': float(np.mean(boot_aris)),
        }
    },
    'clusters': {
        str(cid): {
            'name': cluster_names[cid],
            'size': int((madonna_cleaned['cluster'] == cid).sum()),
            'profile': {f: float(cluster_means.loc[cid, f]) for f in CLUSTER_FEATURES}
        } for cid in range(OPTIMAL_K)
    },
}
with open('exports/web_bundle.json', 'w') as f:
    json.dump(web_bundle, f, indent=2)

print("Exported to ./exports/:")
for fname in sorted(os.listdir('exports')):
    print(f"  - exports/{fname}")


## 13. Export for the Astro frontend (`spotify-analysis.astro`)

The site now imports two files — `cluster_summary.json` and `music_galaxy.json` — from
`public/assets/spotify/` (`diva_dna.json` is gone along with the diva comparison, so the page's
diva-radar section needs to be removed or repurposed on the frontend side too). The old version
of the site hardcoded 4 persona cards (name/description/color/image) directly in the page's JS,
tied to `cluster` being 0-3. `cluster_summary.json` now *carries* that persona metadata
(auto-generated from each cluster's z-scores, same logic as Section 8) so the page reads it
instead of hardcoding it. K is fixed at 4 here, so this happens to still line up with 4 cards,
but the page no longer needs to assume that.

`tsne_x` / `tsne_y` below are UMAP coordinates (Section 9) — kept under the old field names so the
page's galaxy scatter plot doesn't need to change, since it just needs 2D coordinates, not
specifically a t-SNE projection.


In [ ]:
PERSONA_COLORS = ['#FF4D8D', '#22C55E', '#FACC15', '#38BDF8', '#A78BFA', '#FB923C', '#2DD4BF', '#F472B6']
PERSONA_IMAGES = ['/assets/spotify/thediscodynamo-1.jpg', '/assets/spotify/thediscodynamo-2.png',
                   '/assets/spotify/thediscodynamo-3.jpg']  # only 3 exist right now — cycle through them

PERSONA_DESCRIPTIONS = {
    'Danceable': 'built for the dancefloor', 'Understated': 'quietly restrained',
    'Energetic': 'high-intensity and driving', 'Mellow': 'laid-back and unhurried',
    'Uplifting': 'bright, feel-good energy', 'Melancholic': 'wistful and bittersweet',
    'Acoustic': 'organic, unplugged textures', 'Produced': 'polished studio production',
    'Bold': 'loud and larger-than-life', 'Subtle': 'quiet and understated',
    'Talkative': 'speech-forward and rhythmic', 'Melodic': 'led by melody over words',
    'Instrumental': 'largely wordless soundscapes', 'Vocal-driven': 'centered on the vocal',
    'Live-feel': 'raw, performance energy', 'Studio-polished': 'clean studio sheen',
    'Fast-paced': 'quick, propulsive tempo', 'Slow-burn': 'slow, spacious tempo',
}

def make_description(cid):
    top2 = z_scores.loc[cid].abs().sort_values(ascending=False).head(2).index
    phrases = []
    for feat in top2:
        high, low = FEATURE_ADJECTIVES[feat]
        label = high if z_scores.loc[cid, feat] > 0 else low
        phrases.append(PERSONA_DESCRIPTIONS[label])
    return f"Tracks that are {phrases[0]} and {phrases[1]}."

os.makedirs('exports_astro', exist_ok=True)

# ---- cluster_summary.json ----
cluster_summary_records = []
for cid in range(OPTIMAL_K):
    row = cluster_means.loc[cid]
    cluster_summary_records.append({
        'cluster': int(cid),
        'name': cluster_names[cid],
        'description': make_description(cid),
        'color': PERSONA_COLORS[cid % len(PERSONA_COLORS)],
        'image': PERSONA_IMAGES[cid % len(PERSONA_IMAGES)],
        'danceability': float(row['danceability']),
        'energy': float(row['energy']),
        'valence': float(row['valence']),
        'acousticness': float(row['acousticness']),
        'speechiness': float(row['speechiness']),
        'loudness': float(row['loudness']),
        'tempo': float(row['tempo']),
        'instrumentalness': float(row['instrumentalness']),
        'liveness': float(row['liveness']),
        'size': int((madonna_cleaned['cluster'] == cid).sum()),
    })
with open('exports_astro/cluster_summary.json', 'w') as f:
    json.dump(cluster_summary_records, f, indent=2)

# ---- music_galaxy.json (Madonna tracks only, incl. UMAP coords for the galaxy view) ----
music_galaxy_records = []
for i, (_, row) in enumerate(madonna_cleaned.reset_index(drop=True).iterrows()):
    music_galaxy_records.append({
        'name': row['name'],
        'artists': row['artists'],
        'release_year': int(row['release_year']),
        'cluster': int(row['cluster']),
        'popularity': int(row['popularity']),
        'danceability': float(row['danceability']),
        'energy': float(row['energy']),
        'valence': float(row['valence']),
        'acousticness': float(row['acousticness']),
        'speechiness': float(row['speechiness']),
        'loudness': float(row['loudness']),
        'tempo': float(row['tempo']),
        'tsne_x': float(X_umap[i, 0]),
        'tsne_y': float(X_umap[i, 1]),
    })
with open('exports_astro/music_galaxy.json', 'w') as f:
    json.dump(music_galaxy_records, f, indent=2)

print("Astro export complete — copy these into public/assets/spotify/ :")
for fname in sorted(os.listdir('exports_astro')):
    print(f"  - exports_astro/{fname}")
print(f"\nK is fixed at {OPTIMAL_K}, clustered with {CLUSTER_METHOD}. cluster_summary.json carries")
print("persona name/description/color/image, so the page doesn't need cluster count hardcoded.")


## Summary of what changed vs. v1

| Area | v1 | v2 |
|---|---|---|
| Bugs | `madonna_id` undefined crash | fixed, added schema assertions |
| Features | 5 hand-picked | 9, justified by correlation check |
| Scaling | StandardScaler | PowerTransformer, Yeo-Johnson (audio features have outlier tails) |
| K selection | elbow only, arrow points at K=4 | fixed at K=4 (requirement), diagnostics shown for context |
| Clustering model | KMeans only, assumed | KMeans vs. GMM fit at K=4, higher-silhouette model kept |
| Validation | none | KMeans-vs-GMM ARI, bootstrap stability ARI, K-stability sweep, 50-seed stability, feature-set sensitivity ARI, per-track silhouette, GMM covariance-type comparison, UMAP topology sensitivity |
| Persona names | hardcoded strings | auto-derived from cluster z-scores |
| Dim. reduction | t-SNE | UMAP (better for small-N global structure) |
| Diva comparison | 39-artist comparison, written 3x with drifting lists | removed — scope narrowed to Madonna's catalog only |
| Exports | ~15 overlapping files | 3 CSV/JSON + 1 web bundle |
| Frontend data | site hardcoded 4 persona cards tied to K=4 | `cluster_summary.json` carries persona name/desc/color/image; `diva_dna.json` no longer produced |
